# Question 21 - 151. Reverse Words in a String

Given an input string `s`, reverse the order of the **words**.

A **word** is defined as a sequence of non-space characters. The **words** in `s` will be separated by at least one space.

Return *a string of the words in reverse order concatenated by a single space.*

**Note** that `s` may contain leading or trailing spaces or multiple spaces between two words. The returned string should only have a single space separating the words. Do not include any extra spaces.

**Example 1:**

    Input: s = "the sky is blue"
    Output: "blue is sky the"

**Example 2:**

    Input: s = "  hello world  "
    Output: "world hello"
    Explanation: Your reversed string should not contain leading or trailing spaces.

**Example 3:**

    Input: s = "a good   example"
    Output: "example good a"
    Explanation: You need to reduce multiple spaces between two words to a single space in the reversed string.

**Constraints:**

- `1 <= s.length <= 10^4`
- `s` contains English letters (upper-case and lower-case), digits, and spaces `' '`.
- There is **at least one** word in `s`.

**Follow-up:** If the string data type is mutable in your language, can you solve it **in-place** with `O(1)` extra space?

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Tokenise + reverse + join; follow-up: reverse whole, then reverse each word (in place)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (prepend words to a string) | O(n²) | O(n) |
| 2️⃣ Optimized (split, reverse, join) | O(n) | O(n) |
| ➕ Follow-up (in-place on a char array) | O(n) | O(1) extra |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Reverse the **order of words**, not the letters inside them. The output should have single spaces only, with no leading or trailing spaces."*

- Multiple / leading / trailing spaces? → **Yes**, they must be normalised.
- Case and digits are preserved as they are.
- Python strings are **immutable**, so a true O(1)-space in-place solution needs a `list` of characters. I'll mention that before the follow-up.

### Step 1 · Brute force

**Idea:** Scan left to right, extract each word manually, and **prepend** it to a result string: `result = word + " " + result`.

🗣️ *"A naive version extracts words one at a time and prepends each to the result. It reads naturally, but each prepend copies the whole result string, so total work is 1 + 2 + … ≈ O(n²) in the worst case. The fix is to collect words in a list and join once at the end."*

In [1]:
class SolutionBrute:
    def reverseWords(self, s: str) -> str:
        result = ""
        i, n = 0, len(s)
        while i < n:
            while i < n and s[i] == " ":
                i += 1
            if i == n:
                break
            j = i
            while j < n and s[j] != " ":
                j += 1
            word = s[i:j]
            result = word if not result else word + " " + result   # O(len(result)) copy
            i = j
        return result

### Step 2 · Optimize

**Bottleneck:** repeated string concatenation. Strings are immutable, so every `+` allocates and copies.

**Key insight:** gather words into a **list** (append is amortized O(1)), reverse the list, and **`" ".join`** once. `join` computes the total length and copies each character exactly once.

`s.split()` with no argument already handles the messy spacing: it splits on runs of whitespace and drops empties.

🗣️ *"I'll split on whitespace, which also takes care of the extra spaces, then reverse the list and join with single spaces. Every step is linear, so it's O(n) time and O(n) space for the word list and output."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Prepend to a string | ❌ | O(n²) copying because strings are immutable. |
| **`split()` → reverse → `join`** | ✅ | Linear, idiomatic, and handles all the spacing rules for free. |
| `deque.appendleft(word)` then join | ✅ | Also O(n). It's what you'd use if you parse words manually and want to avoid a separate reverse. |
| Stack of words | ✅ | Same as a list reversal. Fine to mention. |
| Regex `re.split(r" +", s.strip())` | ⚠️ | Works but adds overhead and edge cases. `split()` is simpler. |
| **In-place char array (follow-up)** | ✅ for follow-up | Reverse the whole array, then reverse each word, then compact spaces. O(1) extra, but only in languages with mutable strings. |

**Expect the interviewer to ask "without built-ins?".** Then show the in-place three-step version below.

In [2]:
class Solution:
    def reverseWords(self, s: str) -> str:
        return " ".join(reversed(s.split()))


class SolutionInPlace:
    # Follow-up: O(1) extra space on a mutable char array (simulated with a list)
    def reverseWords(self, s: str) -> str:
        a = list(s)

        def reverse(lo: int, hi: int) -> None:
            while lo < hi:
                a[lo], a[hi] = a[hi], a[lo]
                lo += 1
                hi -= 1

        # 1) compact spaces in place: write pointer w, read pointer r
        w = 0
        r = 0
        n = len(a)
        while r < n:
            if a[r] != " ":
                if w > 0:
                    a[w] = " "            # single separator before a new word
                    w += 1
                while r < n and a[r] != " ":
                    a[w] = a[r]
                    w += 1
                    r += 1
            else:
                r += 1
        del a[w:]                         # truncate (in C++/Java: track length)

        # 2) reverse the whole thing: "the sky" -> "yks eht"
        reverse(0, len(a) - 1)

        # 3) reverse each word back:  "yks eht" -> "sky the"
        start = 0
        for i in range(len(a) + 1):
            if i == len(a) or a[i] == " ":
                reverse(start, i - 1)
                start = i + 1
        return "".join(a)

### Step 3 · Toy example walkthrough

**Main solution:** `s = "  hello world  "`

| step | value |
|---|---|
| `s.split()` | `["hello", "world"]` (leading/trailing/multiple spaces gone) |
| `reversed(...)` | `["world", "hello"]` |
| `" ".join(...)` | `"world hello"` ✅ |

**In-place follow-up:** `s = "a good   example"`

| step | char array |
|---|---|
| 1) compact spaces | `a good example` |
| 2) reverse whole | `elpmaxe doog a` |
| 3) reverse each word | `example good a` ✅ |

Why does reverse-all-then-reverse-each work? Reversing the whole string puts the words in the right **order** but spells each one backwards. Reversing each word individually fixes the spelling without changing the order.

In [3]:
cases = [
    ("the sky is blue", "blue is sky the"),
    ("  hello world  ", "world hello"),
    ("a good   example", "example good a"),
    ("single", "single"),
    ("   x   ", "x"),
    ("Alice does not even like Bob", "Bob like even not does Alice"),
    ("1 2 3", "3 2 1"),
]
for s, expected in cases:
    for Impl in (SolutionBrute, Solution, SolutionInPlace):
        assert Impl().reverseWords(s) == expected, (Impl.__name__, s)
    print(repr(s), "->", repr(expected))
print("All tests passed ✅")

'the sky is blue' -> 'blue is sky the'
'  hello world  ' -> 'world hello'
'a good   example' -> 'example good a'
'single' -> 'single'
'   x   ' -> 'x'
'Alice does not even like Bob' -> 'Bob like even not does Alice'
'1 2 3' -> '3 2 1'
All tests passed ✅


### Step 4 · Wrap up

| | Time | Extra space | Why |
|---|---|---|---|
| Prepend to string | O(n²) | O(n) | Each prepend copies the whole result |
| **split / reverse / join** | **O(n)** | **O(n)** | List of words plus output, each char copied a constant number of times |
| In-place char array | O(n) | O(1) | Three linear passes with two pointers |

**Complexity, explained:**
- `split()` reads each character once → O(n). `reversed` is O(#words). `join` writes each character once → O(n). Total O(n).
- Space: the word list and output hold O(n) characters.
- In-place: compacting, the full reverse, and the per-word reverses each touch every character a constant number of times → O(n). We use only a few indices → O(1) extra (in a language with mutable strings).

**Edge cases:** one word, only spaces around one word, multiple internal spaces.

🗣️ **30-second recap:** *"Split on whitespace, which normalises all the spacing, reverse the list of words, and join with single spaces. That's O(n) time and O(n) space. Repeated string concatenation would be O(n²), so I avoid it. For the in-place follow-up on a mutable char array: compact the spaces, reverse the whole array, then reverse each word back. That's O(n) time and O(1) extra space."*